# IFCE Campus Tauá - Tecnologia em Análise e Desenvolvimento de Sistemas (ADS)
## Banco de Dados Não-Relacionais (ADS16) — Aula 05: Redis na Prática
**Prof. Me. Reginaldo Fernandes**

---
### Objetivos Deste Caderno Prático:
1. **Padrão Cache-Aside com TTL:** Implementar leitura e gravação com expiração automática, mensurando a aceleração (*Speedup*) da RAM sobre o disco.
2. **Controle de Sessão e Tokens:** Gerenciar estados efêmeros de autenticação em memória.
3. **Carrinho de Compras Atômico com Hashes:** Manipular itens com `HINCRBY`, `HGETALL` e `HDEL`.
4. **Filas de Processamento com Listas:** Produzir pedidos com `LPUSH` e consumir com `BRPOP`.
5. **Conexão com a Aula 06 (MongoDB):** Antecipar como os dados consumidos da fila se transformarão em documentos BSON persistentes.

### 1. Conexão com o Servidor Redis

In [ ]:
import redis
import json
import time

# Conecta ao Redis local (porta 6379)
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

print("Conexão com Redis ativa:", r.ping())

### 2. Implementando o Padrão Cache-Aside com TTL (Time-To-Live)

In [ ]:
def buscar_catalogo_produtos(chave_cache="cache:produtos_teste"):
    t_inicio = time.time()
    
    # 1. Verifica no Redis
    if r.exists(chave_cache):
        dados = json.loads(r.get(chave_cache))
        latencia = (time.time() - t_inicio) * 1000
        return {"origem": "CACHE_HIT (RAM)", "latencia_ms": round(latencia, 2), "ttl": r.ttl(chave_cache), "dados": dados}
    
    # 2. Se não existir (CACHE MISS), simula leitura lenta de banco/disco
    time.sleep(0.12) # 120 ms de overhead
    dados_db = [
        {"id": 1, "nome": "Notebook Dell Inspiron", "preco": 3899.90},
        {"id": 2, "nome": "Monitor UltraWide LG", "preco": 1249.00}
    ]
    
    # 3. Grava no Redis com TTL de 30 segundos (SETEX)
    r.setex(chave_cache, 30, json.dumps(dados_db))
    latencia = (time.time() - t_inicio) * 1000
    return {"origem": "CACHE_MISS (Disco/DB)", "latencia_ms": round(latencia, 2), "ttl": 30, "dados": dados_db}

# Primeira busca: CACHE MISS esperado
r.delete("cache:produtos_teste")
print("1ª Chamada:", buscar_catalogo_produtos())

# Segunda busca: CACHE HIT imediato em microssegundos!
print("2ª Chamada:", buscar_catalogo_produtos())

### 3. Carrinho de Compras Atômico Usando Hashes do Redis

In [ ]:
session_id = "sessao_aluno_taua_001"
chave_carrinho = f"cart:{session_id}"
r.delete(chave_carrinho)

# HINCRBY adiciona ou incrementa a quantidade do produto de forma atômica
r.hincrby(chave_carrinho, "prod_notebook", 1)
r.hincrby(chave_carrinho, "prod_mouse", 2)
r.hincrby(chave_carrinho, "prod_mouse", 1) # Agora tem 3 mouses

# Define expiração de 1 hora para carrinhos abandonados
r.expire(chave_carrinho, 3600)

# Recupera todo o carrinho com HGETALL
carrinho_atual = r.hgetall(chave_carrinho)
print(f"Conteúdo do Carrinho ({chave_carrinho}):", carrinho_atual)
print(f"TTL restante do carrinho:", r.ttl(chave_carrinho), "segundos")

### 4. Fila Assíncrona de Pedidos: Produtor (LPUSH) e Consumidor (BRPOP)

In [ ]:
chave_fila = "fila:pedidos_lab"
r.delete(chave_fila)

# 1. PRODUTOR: Simula o Checkout da Loja Web enviando um pedido
novo_pedido = {
    "id_pedido": "PED-2026-9999",
    "cliente": "João Silva",
    "total": 4519.90,
    "itens": [{"nome": "Notebook", "qtd": 1}, {"nome": "Mouse", "qtd": 1}]
}

posicao = r.lpush(chave_fila, json.dumps(novo_pedido))
print(f"✓ Pedido enfileirado na lista '{chave_fila}'. Tamanho atual da fila: {posicao}")

# 2. CONSUMIDOR (WORKER): Retira o pedido da fila usando BRPOP
lista, pedido_extraido = r.brpop(chave_fila, timeout=3)
pedido_dict = json.loads(pedido_extraido)
print(f"📥 Pedido consumido com sucesso pelo Worker: {pedido_dict['id_pedido']} - Total: R$ {pedido_dict['total']}")
print("\n>> NA AULA 06: Esse dicionário será salvo diretamente como Documento BSON no MongoDB!")